Обучение BiLSTM+CRF для NER с entity-level метриками


In [1]:
import os
import ast
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from TorchCRF import CRF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from collections import Counter
from tqdm.auto import tqdm

TRAIN_BIO_PATH = "../../data/processed/final_train.csv"
VAL_BIO_PATH = "../../data/processed/final_val.csv"
EMBEDDINGS_PATH = "../../data/external/embeddings/ru_en_aligned.pkl"
MODELS_DIR = "../../models/iteration-2/"
ARTEFACTS_PATH = os.path.join(MODELS_DIR, "artefacts_v2.pkl")
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "bilstm_v2_best.pth")


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Информация: Вычисления будут производиться на устройстве: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Информация: Random seed ({SEED}) установлен для обеспечения воспроизводимости.")
print("\nДиагностика CUDA:")
print(f"- torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"- torch.version.cuda: {getattr(torch.version, 'cuda', None)}")
try:
    from torch.backends import cudnn

    print(
        f"- cudnn.enabled: {cudnn.enabled}, cudnn.version(): {cudnn.version() if hasattr(cudnn, 'version') else None}")
except Exception as e:
    print(f"- Информация по cuDNN недоступна: {e}")
print(f"- torch.cuda.device_count(): {torch.cuda.device_count()}")
for idx in range(torch.cuda.device_count()):
    try:
        print(f"  * [{idx}] {torch.cuda.get_device_name(idx)}")
    except Exception as e:
        print(f"  * [{idx}] ошибка чтения имени устройства: {e}")


Информация: Вычисления будут производиться на устройстве: cpu
Информация: Random seed (42) установлен для обеспечения воспроизводимости.

Диагностика CUDA:
- torch.cuda.is_available(): False
- torch.version.cuda: None
- cudnn.enabled: True, cudnn.version(): None
- torch.cuda.device_count(): 0


Гиперпараметры


In [3]:
WORD_EMBEDDING_DIM = 300
CHAR_EMBEDDING_DIM = 50
CHAR_HIDDEN_DIM = 50
LSTM_HIDDEN_DIM = 256
LSTM_NUM_LAYERS = 2
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20
DROPOUT_RATE = 0.5
EARLY_STOPPING_PATIENCE = 3
RARE_TAGS = ("B-PERCENT", "I-PERCENT", "B-VOLUME", "I-VOLUME")
RARE_SAMPLE_BOOST = 3


Модель


In [4]:
class CharEmbedding(nn.Module):
    def __init__(self, char_vocab_size, embedding_dim, hidden_dim, dropout_rate=0.25):
        super(CharEmbedding, self).__init__()
        self.embedding = nn.Embedding(char_vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, bidirectional=True,
                            batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        print("Информация: Модуль CharEmbedding успешно инициализирован.")

    def forward(self, x):
        batch_size, seq_len, word_len = x.size()
        x = x.view(batch_size * seq_len, word_len)
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        lstm_out, _ = self.lstm(embedded)
        output = lstm_out.permute(0, 2, 1)
        output = torch.max(output, 2)[0]
        output = output.view(batch_size, seq_len, -1)
        return self.dropout(output)


class BiLSTMCrfForNer(nn.Module):
    def __init__(self, word_vocab_size, word_embedding_dim, char_vocab_size, char_embedding_dim, char_hidden_dim,
                 lstm_hidden_dim, num_tags, dropout_rate=0.33, padding_idx=0):
        super(BiLSTMCrfForNer, self).__init__()
        self.word_embedding = nn.Embedding(num_embeddings=word_vocab_size, embedding_dim=word_embedding_dim,
                                           padding_idx=padding_idx)
        self.word_embedding.weight.requires_grad = True
        self.char_embedding = CharEmbedding(char_vocab_size=char_vocab_size, embedding_dim=char_embedding_dim,
                                            hidden_dim=char_hidden_dim, dropout_rate=dropout_rate)
        self.embedding_dropout = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(input_size=word_embedding_dim + (2 * char_hidden_dim), hidden_size=lstm_hidden_dim,
                            num_layers=2, bidirectional=True, batch_first=True, dropout=dropout_rate if 2 > 1 else 0)
        self.classifier = nn.Linear(2 * lstm_hidden_dim, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)
        print("Информация: Основная модель BiLSTMCrfForNer успешно инициализирована.")

    def forward(self, word_ids, char_ids, mask, tags=None):
        word_embeds = self.word_embedding(word_ids)
        char_embeds = self.char_embedding(char_ids)
        combined_embeds = torch.cat([word_embeds, char_embeds], dim=-1)
        combined_embeds = self.embedding_dropout(combined_embeds)
        lstm_out, _ = self.lstm(combined_embeds)

        emissions = self.classifier(lstm_out)
        mask = mask.bool()
        if tags is not None:
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return loss
        else:
            decoded_tags = self.crf.decode(emissions, mask=mask)
            return decoded_tags


Данные


In [5]:
class NerDataset(Dataset):
    def __init__(self, df_path, word2id=None, char2id=None, tag2id=None):
        self.df = pd.read_csv(df_path, sep=";")
        self.df = pd.read_csv(df_path, sep=";")
        self.df["tokens"] = self.df["tokens"].apply(ast.literal_eval)
        self.df["tags"] = self.df["tags"].apply(ast.literal_eval)

        # Согласованная нормализация токенов для уменьшения OOV
        def tok_norm(t: str) -> str:
            t2 = t.strip().lower()
            t2 = "".join("0" if ch.isdigit() else ch for ch in t2)
            t2 = t2.replace("％", "%").replace("ℓ", "л").replace("l", "л")
            return t2

        self.tok_norm_fn = tok_norm

        if word2id is None:
            # строим словарь по нормализованным токенам
            self.word2id, self.id2word = self._build_vocab(self.df["tokens"], norm_fn=self.tok_norm_fn)
        else:
            self.word2id, self.id2word = word2id, {v: k for k, v in word2id.items()}

        if char2id is None:
            all_chars = set("".join(["".join(tokens) for tokens in self.df["tokens"]]))
            self.char2id, self.id2char = self._build_char_vocab(all_chars)
        else:
            self.char2id, self.id2char = char2id, {v: k for k, v in char2id.items()}

        if tag2id is None:
            self.tag2id, self.id2tag = self._build_tag_vocab(self.df["tags"])
        else:
            self.tag2id, self.id2tag = tag2id, {v: k for k, v in tag2id.items()}
        print(f"Информация: Dataset загружен. Размер: {len(self.df)} записей.")
        print(f"Информация: Размер словаря слов: {len(self.word2id)}")
        print(f"Информация: Размер словаря символов: {len(self.char2id)}")
        print(f"Информация: Размер словаря тегов: {len(self.tag2id)}")

    def _build_vocab(self, data, norm_fn=lambda x: x):
        vocab = {"<PAD>": 0, "<UNK>": 1}
        for sequence in data:
            for item in sequence:
                key = norm_fn(item)
                if key not in vocab:
                    vocab[key] = len(vocab)
        id2vocab = {v: k for k, v in vocab.items()}
        return vocab, id2vocab

    def _build_char_vocab(self, chars):
        vocab = {"<PAD>": 0, "<UNK>": 1}
        for char in sorted(list(chars)):
            if char not in vocab:
                vocab[char] = len(vocab)
        id2vocab = {v: k for k, v in vocab.items()}
        return vocab, id2vocab

    def _build_tag_vocab(self, tags_series):
        tag_vocab = {"<PAD>": 0}
        if "O" not in tag_vocab:
            tag_vocab["O"] = len(tag_vocab)
        for seq in tags_series:
            for tag in seq:
                if tag not in tag_vocab:
                    tag_vocab[tag] = len(tag_vocab)
        id2tag = {v: k for k, v in tag_vocab.items()}
        return tag_vocab, id2tag

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = row["tokens"]
        tags = row["tags"]
        # используем ту же нормализацию, что и при построении словаря
        word_ids = [self.word2id.get(self.tok_norm_fn(token), self.word2id["<UNK>"]) for token in tokens]
        o_idx = self.tag2id.get("O", 0)
        tag_ids = [self.tag2id.get(tag, o_idx) for tag in tags]
        char_ids = []
        for token in tokens:
            ids = [self.char2id.get(char, self.char2id["<UNK>"]) for char in token]
            char_ids.append(ids)
        return {"words": word_ids, "chars": char_ids, "tags": tag_ids}

Метрики entity-level


In [6]:
def _extract_entities_bio(tags_seq):
    entities = set()
    cur_type = None
    start = None

    def close_entity(end_idx):
        nonlocal cur_type, start
        if cur_type is not None and start is not None:
            entities.add((cur_type, start, end_idx))
        cur_type, start = None, None

    for i, tag in enumerate(tags_seq):
        if tag in ("O", "<PAD>", "<UNK>") or not isinstance(tag, str):
            if cur_type is not None:
                close_entity(i - 1)
            continue
        if tag.startswith("B-"):
            if cur_type is not None:
                close_entity(i - 1)
            cur_type = tag[2:]
            start = i
        elif tag.startswith("I-"):
            t = tag[2:]
            if cur_type == t and start is not None:
                pass
            else:
                if cur_type is not None:
                    close_entity(i - 1)
                cur_type = t
                start = i
        else:
            if cur_type is not None:
                close_entity(i - 1)
    if cur_type is not None:
        close_entity(len(tags_seq) - 1)
    return entities


def _compute_entity_report(all_true_entities, all_pred_entities):
    per_type = {}
    support_per_type = Counter()
    total_tp = total_fp = total_fn = 0
    for true_set, pred_set in zip(all_true_entities, all_pred_entities):
        types = set([t for (t, _, _) in true_set]) | set([t for (t, _, _) in pred_set])
        for t in types:
            true_t = {e for e in true_set if e[0] == t}
            pred_t = {e for e in pred_set if e[0] == t}
            tp = len(true_t & pred_t)
            fp = len(pred_t - true_t)
            fn = len(true_t - pred_t)
            if t not in per_type:
                per_type[t] = {"tp": 0, "fp": 0, "fn": 0}
            per_type[t]["tp"] += tp
            per_type[t]["fp"] += fp
            per_type[t]["fn"] += fn
            support_per_type[t] += len(true_t)
            total_tp += tp
            total_fp += fp
            total_fn += fn
    report = {}
    for t, c in per_type.items():
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        report[t] = {"precision": prec, "recall": rec, "f1-score": f1, "support": support_per_type[t]}
    micro_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = 2 * micro_prec * micro_rec / (micro_prec + micro_rec) if (micro_prec + micro_rec) > 0 else 0.0
    report["micro avg"] = {"precision": micro_prec, "recall": micro_rec, "f1-score": micro_f1,
                           "support": sum(support_per_type.values())}
    valid_types = [t for t in report.keys() if t not in ("micro avg", "macro avg", "weighted avg")]
    if valid_types:
        macro_prec = sum(report[t]["precision"] for t in valid_types) / len(valid_types)
        macro_rec = sum(report[t]["recall"] for t in valid_types) / len(valid_types)
        macro_f1 = sum(report[t]["f1-score"] for t in valid_types) / len(valid_types)
    else:
        macro_prec = macro_rec = macro_f1 = 0.0
    report["macro avg"] = {"precision": macro_prec, "recall": macro_rec, "f1-score": macro_f1,
                           "support": sum(support_per_type.values())}
    total_support = sum(support_per_type.values())
    if total_support > 0:
        weighted_prec = sum(report[t]["precision"] * support_per_type[t] for t in valid_types) / total_support
        weighted_rec = sum(report[t]["recall"] * support_per_type[t] for t in valid_types) / total_support
        weighted_f1 = sum(report[t]["f1-score"] * support_per_type[t] for t in valid_types) / total_support
    else:
        weighted_prec = weighted_rec = weighted_f1 = 0.0
    report["weighted avg"] = {"precision": weighted_prec, "recall": weighted_rec, "f1-score": weighted_f1,
                              "support": total_support}
    return report


def eval_epoch_entities(model, dataloader, id2tag):
    model.eval()
    all_true_entities = []
    all_pred_entities = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Eval", leave=False, dynamic_ncols=True):
            words = batch["words"].to(device)
            chars = batch["chars"].to(device)
            tags = batch["tags"].to(device)
            mask = batch["mask"].to(device)
            predictions = model(words, chars, mask)
            for i in range(len(predictions)):
                seq_len = mask[i].sum().item()
                true_ids = tags[i][:seq_len].cpu().tolist()
                pred_ids = predictions[i][:seq_len]
                true_tags = [id2tag[idx] for idx in true_ids]
                pred_tags = [id2tag[idx] for idx in pred_ids]
                true_ents = _extract_entities_bio(true_tags)
                pred_ents = _extract_entities_bio(pred_tags)
                all_true_entities.append(true_ents)
                all_pred_entities.append(pred_ents)
    return _compute_entity_report(all_true_entities, all_pred_entities)


Эмбеддинги


In [7]:
def load_and_prepare_embeddings(word2id, filepath, embedding_dim):
    with open(filepath, "rb") as f:
        fasttext_model = pickle.load(f)

    def normalize_token(tok: str) -> str:
        t = tok.strip().lower()
        t = "".join("0" if ch.isdigit() else ch for ch in t)
        t = t.replace("％", "%").replace("ℓ", "л").replace("l", "л")
        return t

    def lookup_vector(word: str):
        candidates = [
            word,
            word.lower(),
            normalize_token(word),
            normalize_token(word).replace(" ", ""),
        ]
        for c in candidates:
            if c in fasttext_model:
                return fasttext_model[c]
        return None

    embedding_matrix = np.random.uniform(-0.05, 0.05, (len(word2id), embedding_dim))
    hits = 0
    tried_norm_hit = 0
    for word, i in word2id.items():
        vec = lookup_vector(word)
        if vec is not None:
            embedding_matrix[i] = vec
            hits += 1
        else:
            simple = "".join(ch for ch in normalize_token(word) if ch.isalnum() or ch in ["%", "л"])
            if simple in fasttext_model:
                embedding_matrix[i] = fasttext_model[simple]
                hits += 1
                tried_norm_hit += 1

    print(
        f"Информация: Найдено {hits} из {len(word2id)} слов в предобученной модели ({hits / len(word2id) * 100:.2f}%). (доп. за счёт нормализации: {tried_norm_hit})")
    embedding_matrix[word2id["<PAD>"]] = np.zeros(embedding_dim)
    return torch.tensor(embedding_matrix, dtype=torch.float)

Датасеты и DataLoader


In [8]:
def collate_fn(batch, word_pad_idx=0, char_pad_idx=0, tag_pad_idx=0):
    max_seq_len = max(len(item["words"]) for item in batch)
    max_word_len = max(
        max(len(char_seq) for char_seq in item["chars"]) if item["chars"] else 0
        for item in batch
    )
    padded_words, padded_chars, padded_tags, masks = [], [], [], []
    for item in batch:
        seq_len = len(item["words"])
        padded_words.append(item["words"] + [word_pad_idx] * (max_seq_len - seq_len))
        padded_tags.append(item["tags"] + [tag_pad_idx] * (max_seq_len - seq_len))
        masks.append([1] * seq_len + [0] * (max_seq_len - seq_len))
        padded_char_seq = []
        for char_seq in item["chars"]:
            padded_char_seq.append(char_seq + [char_pad_idx] * (max_word_len - len(char_seq)))
        if seq_len < max_seq_len:
            for _ in range(max_seq_len - seq_len):
                padded_char_seq.append([char_pad_idx] * max_word_len)
        padded_chars.append(padded_char_seq)
    return {
        "words": torch.tensor(padded_words, dtype=torch.long),
        "chars": torch.tensor(padded_chars, dtype=torch.long),
        "tags": torch.tensor(padded_tags, dtype=torch.long),
        "mask": torch.tensor(masks, dtype=torch.bool),
    }

In [9]:
train_dataset = NerDataset(TRAIN_BIO_PATH)
val_dataset = NerDataset(VAL_BIO_PATH, word2id=train_dataset.word2id, char2id=train_dataset.char2id,
                         tag2id=train_dataset.tag2id)


def build_sample_weights(dataset, rare_tags=RARE_TAGS, base_weight=1.0, rare_boost=RARE_SAMPLE_BOOST):
    weights = []
    for tags in dataset.df["tags"]:
        has_rare = any(t in rare_tags for t in tags)
        weights.append(rare_boost if has_rare else base_weight)
    return torch.tensor(weights, dtype=torch.float)


print("Информация: Пересчет весов для WeightedRandomSampler на новом датасете...")
train_weights = build_sample_weights(train_dataset, rare_tags=RARE_TAGS, rare_boost=RARE_SAMPLE_BOOST)

train_sampler = WeightedRandomSampler(weights=train_weights, num_samples=len(train_weights), replacement=True)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("\nИнформация: Датасеты и даталоадеры успешно созданы.")

embedding_weights = load_and_prepare_embeddings(train_dataset.word2id, EMBEDDINGS_PATH, WORD_EMBEDDING_DIM)


# Диагностика покрытия на train/val
def coverage_report(dataset, word2id, embedding_tensor):
    emb = embedding_tensor.cpu().numpy()
    pad_id = word2id["<PAD>"]
    # считаем строку-хит, если не вся из нулей (кроме PAD, которому мы специально поставили нули)
    hits_mask = (np.abs(emb).sum(axis=1) > 0)
    hits_mask[pad_id] = False

    total_tokens, hit_tokens = 0, 0
    for tokens in dataset.df["tokens"]:
        for t in tokens:
            t2 = dataset.tok_norm_fn(t) if hasattr(dataset, "tok_norm_fn") else t
            tok_id = word2id.get(t2, word2id["<UNK>"])
            if tok_id == pad_id:
                continue
            total_tokens += 1
            if 0 <= tok_id < len(hits_mask) and hits_mask[tok_id]:
                hit_tokens += 1
    coverage = (hit_tokens / total_tokens) * 100 if total_tokens else 0.0
    print(f"Покрытие эмбеддингов: {coverage:.2f}% ({hit_tokens}/{total_tokens})")


print("\nДиагностика покрытия:")
coverage_report(train_dataset, train_dataset.word2id, embedding_weights)
coverage_report(val_dataset, train_dataset.word2id, embedding_weights)

Информация: Dataset загружен. Размер: 42639 записей.
Информация: Размер словаря слов: 6560
Информация: Размер словаря символов: 151
Информация: Размер словаря тегов: 10
Информация: Dataset загружен. Размер: 7524 записей.
Информация: Размер словаря слов: 6560
Информация: Размер словаря символов: 151
Информация: Размер словаря тегов: 10
Информация: Пересчет весов для WeightedRandomSampler на новом датасете...

Информация: Датасеты и даталоадеры успешно созданы.
Информация: Найдено 5498 из 6560 слов в предобученной модели (83.81%). (доп. за счёт нормализации: 2292)

Диагностика покрытия:
Покрытие эмбеддингов: 100.00% (145897/145897)
Покрытие эмбеддингов: 100.00% (25742/25742)


Инициализация модели


In [10]:
if embedding_weights is not None:
    model = BiLSTMCrfForNer(
        word_vocab_size=len(train_dataset.word2id),
        word_embedding_dim=WORD_EMBEDDING_DIM,
        char_vocab_size=len(train_dataset.char2id),
        char_embedding_dim=CHAR_EMBEDDING_DIM,
        char_hidden_dim=CHAR_HIDDEN_DIM,
        lstm_hidden_dim=LSTM_HIDDEN_DIM,
        num_tags=len(train_dataset.tag2id),
        dropout_rate=DROPOUT_RATE,
        padding_idx=train_dataset.word2id["<PAD>"],
    )
    model.word_embedding.weight.data.copy_(embedding_weights)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",  # используйте "min", если мониторите val_loss
        factor=0.5,
        patience=1,
        min_lr=1e-6
    )
    print("\nИнформация: Модель и оптимизатор инициализированы.")


Информация: Модуль CharEmbedding успешно инициализирован.
Информация: Основная модель BiLSTMCrfForNer успешно инициализирована.

Информация: Модель и оптимизатор инициализированы.


Обучение


In [11]:
def train_epoch(model, dataloader, optimizer):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataloader, desc="Train", leave=False, dynamic_ncols=True)
    for batch in pbar:
        words = batch["words"].to(device)
        chars = batch["chars"].to(device)
        tags = batch["tags"].to(device)
        mask = batch["mask"].to(device)
        optimizer.zero_grad()
        loss = model(words, chars, mask, tags)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(dataloader)


best_val_f1 = 0.0
best_model_state = None
no_improve = 0

print("\n--- Начало процесса обучения ---")
for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc="Epochs", unit="epoch", dynamic_ncols=True):
    train_loss = train_epoch(model, train_dataloader, optimizer)
    report = eval_epoch_entities(model, val_dataloader, train_dataset.id2tag)
    val_f1_macro = report["macro avg"]["f1-score"]
    old_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_f1_macro)
    new_lr = optimizer.param_groups[0]["lr"]
    if new_lr < old_lr:
        print(f"ReduceLROnPlateau: lr уменьшен с {old_lr:.2e} до {new_lr:.2e}")
    print(f"\nЭпоха {epoch}/{NUM_EPOCHS}:")
    print(f"  Потери на обучении (Train Loss): {train_loss:.4f}")
    print(f"  F1-macro (entity-level) на валидации: {val_f1_macro:.4f}")
    print("  Детальный отчёт по сущностям:")
    for tag, metrics in report.items():
        if isinstance(metrics, dict) and tag not in {"micro avg", "macro avg", "weighted avg"}:
            print(
                f"    - {tag:<10}: F1={metrics['f1-score']:.4f}, Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, Support={metrics['support']}")
    print("  Агрегаты:")
    for agg in ("micro avg", "macro avg", "weighted avg"):
        m = report[agg]
        print(
            f"    - {agg:<12}: F1={m['f1-score']:.4f}, Precision={m['precision']:.4f}, Recall={m['recall']:.4f}, Support={m['support']}")
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        best_model_state = model.state_dict().copy()
        print(f"  Новый лучший результат! Модель сохранена (F1-macro entity-level: {best_val_f1:.4f}).")
        os.makedirs(MODELS_DIR, exist_ok=True)
        torch.save(best_model_state, BEST_MODEL_PATH)
        no_improve = 0
    else:
        no_improve += 1
        print(f"  Нет улучшения {no_improve}/{EARLY_STOPPING_PATIENCE}.")
        if no_improve >= EARLY_STOPPING_PATIENCE:
            print("  Ранняя остановка.")
            break
print("\n--- Обучение завершено ---")
print(f"Лучший F1-macro на валидации: {best_val_f1:.4f}")



--- Начало процесса обучения ---


Epochs:   0%|          | 0/20 [00:00<?, ?epoch/s]

Train:   0%|          | 0/1333 [00:00<?, ?it/s]

Eval:   0%|          | 0/236 [00:00<?, ?it/s]


Эпоха 1/20:
  Потери на обучении (Train Loss): 0.7426
  F1-macro (entity-level) на валидации: 0.9278
  Детальный отчёт по сущностям:
    - TYPE      : F1=0.9617, Precision=0.9393, Recall=0.9853, Support=7753
    - BRAND     : F1=0.7808, Precision=0.8586, Recall=0.7160, Support=1264
    - VOLUME    : F1=0.9910, Precision=0.9882, Recall=0.9938, Support=3383
    - PERCENT   : F1=0.9777, Precision=0.9981, Recall=0.9582, Support=550
  Агрегаты:
    - micro avg   : F1=0.9539, Precision=0.9479, Recall=0.9601, Support=12950
    - macro avg   : F1=0.9278, Precision=0.9461, Recall=0.9133, Support=12950
    - weighted avg: F1=0.9524, Precision=0.9467, Recall=0.9601, Support=12950
  Новый лучший результат! Модель сохранена (F1-macro entity-level: 0.9278).


...

Eval:   0%|          | 0/236 [00:00<?, ?it/s]

ReduceLROnPlateau: lr уменьшен с 5.00e-04 до 2.50e-04

Эпоха 10/20:
  Потери на обучении (Train Loss): 0.0398
  F1-macro (entity-level) на валидации: 0.9635
  Детальный отчёт по сущностям:
    - TYPE      : F1=0.9747, Precision=0.9676, Recall=0.9819, Support=7753
    - BRAND     : F1=0.8835, Precision=0.8818, Recall=0.8853, Support=1264
    - VOLUME    : F1=0.9975, Precision=0.9982, Recall=0.9967, Support=3383
    - PERCENT   : F1=0.9982, Precision=1.0000, Recall=0.9964, Support=550
  Агрегаты:
    - micro avg   : F1=0.9727, Precision=0.9685, Recall=0.9770, Support=12950
    - macro avg   : F1=0.9635, Precision=0.9619, Recall=0.9651, Support=12950
    - weighted avg: F1=0.9728, Precision=0.9686, Recall=0.9770, Support=12950
  Нет улучшения 2/3.


Train:   0%|          | 0/1333 [00:00<?, ?it/s]

Eval:   0%|          | 0/236 [00:00<?, ?it/s]


Эпоха 11/20:
  Потери на обучении (Train Loss): 0.0322
  F1-macro (entity-level) на валидации: 0.9627
  Детальный отчёт по сущностям:
    - TYPE      : F1=0.9754, Precision=0.9710, Recall=0.9799, Support=7753
    - BRAND     : F1=0.8819, Precision=0.8833, Recall=0.8805, Support=1264
    - VOLUME    : F1=0.9970, Precision=0.9976, Recall=0.9965, Support=3383
    - PERCENT   : F1=0.9964, Precision=0.9982, Recall=0.9945, Support=550
  Агрегаты:
    - micro avg   : F1=0.9728, Precision=0.9706, Recall=0.9751, Support=12950
    - macro avg   : F1=0.9627, Precision=0.9625, Recall=0.9629, Support=12950
    - weighted avg: F1=0.9728, Precision=0.9705, Recall=0.9751, Support=12950
  Нет улучшения 3/3.
  Ранняя остановка.

--- Обучение завершено ---
Лучший F1-macro на валидации: 0.9646


Сохранение артефактов


In [12]:
os.makedirs(MODELS_DIR, exist_ok=True)
if best_model_state is not None and not os.path.exists(BEST_MODEL_PATH):
    torch.save(best_model_state, BEST_MODEL_PATH)

artefacts = {
    "word2id": train_dataset.word2id,
    "char2id": train_dataset.char2id,
    "tag2id": train_dataset.tag2id,
    "id2tag": train_dataset.id2tag,
}
with open(ARTEFACTS_PATH, "wb") as f:
    pickle.dump(artefacts, f)

print(f"Информация: Лучшая модель сохранена в: {BEST_MODEL_PATH}")
print(f"Информация: Артефакты сохранены в: {ARTEFACTS_PATH}")

Информация: Лучшая модель сохранена в: ../../models/iteration-2/bilstm_v2_best.pth
Информация: Артефакты сохранены в: ../../models/iteration-2/artefacts_v2.pkl
